# Notebook 02 — Group-Safe Final Split
Reviews duplicate/family grouping quality and creates a group-safe train/val/test split.
No model training. No dataset modification.


## Configuration


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import math, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUTPUT_DIR  = Path(r'D:\DIABETES\diabetes_pipeline_outputs')
CONTACT_DIR = OUTPUT_DIR / 'contact_sheets_notebook02'
CONTACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
SPLIT_METHOD = 'group_stratified_custom_seed42'

# Try both possible manifest paths
MANIFEST_CANDIDATES = [
    OUTPUT_DIR / 'outputs' / '01_master_manifest.csv',
    OUTPUT_DIR / '01_master_manifest.csv',
]
MANIFEST_PATH = None
for p in MANIFEST_CANDIDATES:
    if p.exists():
        MANIFEST_PATH = p
        break

if MANIFEST_PATH is None:
    raise FileNotFoundError(
        f'01_master_manifest.csv not found. Checked:\n' +
        '\n'.join(str(p) for p in MANIFEST_CANDIDATES)
    )

print(f'Manifest path : {MANIFEST_PATH}')
print(f'Output dir    : {OUTPUT_DIR}')


Manifest path : D:\DIABETES\diabetes_pipeline_outputs\01_master_manifest.csv
Output dir    : D:\DIABETES\diabetes_pipeline_outputs


## Load and Validate Manifest


In [2]:
REQUIRED_COLS = [
    'image_id','file_path','dataset_source','original_split',
    'folder_label','final_label','label_binary',
    'filename','stem','extension','width','height','image_mode',
    'file_size_bytes','readable','exact_hash_md5','perceptual_hash_phash',
    'filename_family_id','exact_duplicate_group_id','near_duplicate_group_id',
    'effective_group_id','audit_status','audit_notes'
]

df_raw = pd.read_csv(MANIFEST_PATH)
print(f'Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns.')

missing_cols = [c for c in REQUIRED_COLS if c not in df_raw.columns]
if 'effective_group_id' in missing_cols:
    raise ValueError('effective_group_id column is missing — cannot proceed.')
if missing_cols:
    print(f'WARNING: optional columns missing: {missing_cols}')

# Filter to readable, non-excluded images
df = df_raw[
    (df_raw['readable'] == True) &
    (df_raw['audit_status'] != 'excluded_unreadable')
].copy().reset_index(drop=True)

excluded_unreadable = len(df_raw) - len(df)
print(f'Excluded (unreadable): {excluded_unreadable}')
print(f'Working set          : {len(df)} images')


Loaded 2750 rows, 23 columns.
Excluded (unreadable): 0
Working set          : 2750 images


## Group-Level Analysis


In [3]:
grp = df.groupby('effective_group_id').agg(
    group_size       = ('image_id', 'count'),
    labels_in_group  = ('final_label', lambda x: ','.join(sorted(x.unique()))),
    label_binary_vals= ('label_binary', lambda x: sorted(x.unique())),
    original_splits  = ('original_split', lambda x: ','.join(sorted(x.unique()))),
    sample_filenames = ('filename', lambda x: ','.join(list(x)[:3])),
    sample_file_paths= ('file_path', lambda x: ','.join(list(x)[:3])),
).reset_index()

grp['label_conflict'] = grp['label_binary_vals'].apply(lambda v: len(v) > 1)
grp['spans_splits']   = grp['original_splits'].apply(lambda s: ',' in s)

single_img = (grp['group_size'] == 1).sum()
multi_img  = (grp['group_size'] >  1).sum()
conflict   = grp['label_conflict'].sum()
spans      = grp['spans_splits'].sum()

print(f'Total effective groups  : {len(grp)}')
print(f'Single-image groups     : {single_img}')
print(f'Multi-image groups      : {multi_img}')
print(f'Largest group size      : {grp["group_size"].max()}')
print(f'Label-conflict groups   : {conflict}')
print(f'Groups spanning splits  : {spans}')


Total effective groups  : 1314
Single-image groups     : 359
Multi-image groups      : 955
Largest group size      : 5
Label-conflict groups   : 0
Groups spanning splits  : 26


## Detect and Exclude Label-Conflict Groups


In [4]:
conflict_grp_ids = set(grp[grp['label_conflict']]['effective_group_id'])

# Save conflict report
conflict_grps = grp[grp['label_conflict']].copy()
if len(conflict_grps) > 0:
    # Join back to get per-image details
    conflict_detail = df[df['effective_group_id'].isin(conflict_grp_ids)].copy()
    conflict_agg = conflict_detail.groupby('effective_group_id').agg(
        group_size       = ('image_id','count'),
        labels_in_group  = ('final_label', lambda x: ','.join(sorted(x.unique()))),
        diabetes_count   = ('label_binary', lambda x: (x==1).sum()),
        non_diabetes_count=('label_binary', lambda x: (x==0).sum()),
        original_splits_in_group=('original_split', lambda x: ','.join(sorted(x.unique()))),
        file_paths       = ('file_path', lambda x: ','.join(x)),
        filenames        = ('filename', lambda x: ','.join(x)),
    ).reset_index()
    conflict_agg.to_csv(OUTPUT_DIR / '02_label_conflict_groups.csv', index=False)
    print(f'Label-conflict groups: {len(conflict_agg)} — saved.')
else:
    pd.DataFrame(columns=['effective_group_id','group_size','labels_in_group',
                           'diabetes_count','non_diabetes_count',
                           'original_splits_in_group','file_paths','filenames']
                ).to_csv(OUTPUT_DIR / '02_label_conflict_groups.csv', index=False)
    print('No label-conflict groups found.')

# Working set after conflict exclusion
df_clean = df[~df['effective_group_id'].isin(conflict_grp_ids)].copy().reset_index(drop=True)
excluded_conflict = len(df) - len(df_clean)
print(f'Excluded (label conflict): {excluded_conflict} images')
print(f'Working set after exclusions: {len(df_clean)} images')


No label-conflict groups found.
Excluded (label conflict): 0 images
Working set after exclusions: 2750 images


## Save Group Review Reports


In [5]:
# 1. Group review summary
conflict_images = len(df[df['effective_group_id'].isin(conflict_grp_ids)])
span_images     = len(df[df['effective_group_id'].isin(grp[grp['spans_splits']]['effective_group_id'])])

review_summary = pd.DataFrame([{
    'total_images': len(df),
    'total_effective_groups': len(grp),
    'single_image_groups': int(single_img),
    'multi_image_groups': int(multi_img),
    'largest_group_size': int(grp['group_size'].max()),
    'groups_with_label_conflict': int(conflict),
    'images_in_label_conflict_groups': int(conflict_images),
    'groups_spanning_original_splits': int(spans),
    'images_in_groups_spanning_original_splits': int(span_images),
}])
review_summary.to_csv(OUTPUT_DIR / '02_group_review_summary.csv', index=False)
print('Saved: 02_group_review_summary.csv')

# 2. Largest groups
largest = grp.nlargest(50, 'group_size')[[
    'effective_group_id','group_size','labels_in_group',
    'original_splits','sample_filenames','sample_file_paths'
]].rename(columns={'original_splits':'original_splits_in_group'})
largest.to_csv(OUTPUT_DIR / '02_largest_groups.csv', index=False)
print('Saved: 02_largest_groups.csv')

# 3. Original split leakage risk
leakage_risk_grps = grp[grp['spans_splits']]
if len(leakage_risk_grps) > 0:
    leak_detail = df[df['effective_group_id'].isin(leakage_risk_grps['effective_group_id'])]
    leak_agg = leak_detail.groupby('effective_group_id').agg(
        group_size=('image_id','count'),
        labels_in_group=('final_label', lambda x: ','.join(sorted(x.unique()))),
        original_splits_in_group=('original_split', lambda x: ','.join(sorted(x.unique()))),
        file_paths=('file_path', lambda x: ','.join(x)),
        filenames=('filename', lambda x: ','.join(x)),
    ).reset_index()
    leak_agg.to_csv(OUTPUT_DIR / '02_original_split_leakage_risk.csv', index=False)
    print(f'Saved: 02_original_split_leakage_risk.csv ({len(leak_agg)} groups)')
else:
    pd.DataFrame().to_csv(OUTPUT_DIR / '02_original_split_leakage_risk.csv', index=False)
    print('No original-split leakage risk groups found.')


Saved: 02_group_review_summary.csv
Saved: 02_largest_groups.csv
Saved: 02_original_split_leakage_risk.csv (26 groups)


## Group-Safe Train/Val/Test Split


In [6]:
rng = np.random.default_rng(RANDOM_SEED)

# Collapse to group level
grp_clean = df_clean.groupby('effective_group_id').agg(
    group_size  = ('image_id','count'),
    group_label = ('final_label', lambda x: x.iloc[0]),
    label_binary= ('label_binary', 'first'),
).reset_index()

def split_class_groups(group_df, train_r, val_r, seed):
    rng_local = np.random.default_rng(seed)
    ids = group_df['effective_group_id'].tolist()
    rng_local.shuffle(ids)
    n = len(ids)
    n_train = round(n * train_r)
    n_val   = round(n * val_r)
    return ids[:n_train], ids[n_train:n_train+n_val], ids[n_train+n_val:]

split_map = {}
for label_val in [1, 0]:
    cls_groups = grp_clean[grp_clean['label_binary'] == label_val]
    tr_ids, va_ids, te_ids = split_class_groups(
        cls_groups, TRAIN_RATIO, VAL_RATIO, RANDOM_SEED + label_val
    )
    for gid in tr_ids: split_map[gid] = 'train'
    for gid in va_ids: split_map[gid] = 'val'
    for gid in te_ids: split_map[gid] = 'test'

grp_clean['final_split'] = grp_clean['effective_group_id'].map(split_map)

group_meta = grp_clean[['effective_group_id','group_size','group_label','final_split']].copy()
df_split = df_clean.merge(group_meta, on='effective_group_id', how='left')
df_split['split_method'] = SPLIT_METHOD
df_split['split_seed']   = RANDOM_SEED
df_split['group_status'] = 'ok'

print('Split complete.')
print(df_split.groupby(['final_split','final_label']).size().to_string())

Split complete.
final_split  final_label 
test         diabetes        215
             non_diabetes    197
train        diabetes        961
             non_diabetes    969
val          diabetes        199
             non_diabetes    209


## Leakage Check


In [7]:
leak_check = df_split.groupby('effective_group_id')['final_split'].nunique()
leaking = leak_check[leak_check > 1]
leakage_pass = len(leaking) == 0

leakage_result = pd.DataFrame([{
    'groups_appearing_in_multiple_final_splits': len(leaking),
    'leaking_group_ids': ','.join(leaking.index.tolist()) if len(leaking) > 0 else 'none',
    'status': 'PASS' if leakage_pass else 'FAIL'
}])
leakage_result.to_csv(OUTPUT_DIR / '02_final_split_leakage_check.csv', index=False)
print(f'Leakage check: {"PASS" if leakage_pass else "FAIL"}')
if not leakage_pass:
    print(f'WARNING: {len(leaking)} groups leak across splits!')


Leakage check: PASS


## Save All Output Files


In [8]:
# 5. Group-safe split manifest (main output)
MANIFEST_COLS = [
    'image_id','file_path','dataset_source','original_split',
    'folder_label','final_label','label_binary',
    'filename','stem','extension','width','height','image_mode',
    'file_size_bytes','readable','exact_hash_md5','perceptual_hash_phash',
    'filename_family_id','exact_duplicate_group_id','near_duplicate_group_id',
    'effective_group_id','audit_status','audit_notes',
    'final_split','split_method','split_seed','group_size','group_label','group_status'
]
existing_cols = [c for c in MANIFEST_COLS if c in df_split.columns]
df_split[existing_cols].to_csv(OUTPUT_DIR / '02_group_safe_split_manifest.csv', index=False)
print(f'Saved: 02_group_safe_split_manifest.csv ({len(df_split)} rows)')

# 6. Final split summary
split_summary = df_split.groupby(['final_split','final_label','label_binary']).size().reset_index(name='count')
total = split_summary['count'].sum()
split_summary['pct'] = (split_summary['count'] / total * 100).round(2)
split_summary.to_csv(OUTPUT_DIR / '02_final_split_summary.csv', index=False)
print('Saved: 02_final_split_summary.csv')

# 7. Final split group summary
grp_split_summary = grp_clean.groupby(['final_split','group_label']).size().reset_index(name='group_count')
grp_split_summary.to_csv(OUTPUT_DIR / '02_final_split_group_summary.csv', index=False)
print('Saved: 02_final_split_group_summary.csv')

# 9. Original vs final split crosstab
crosstab = pd.crosstab(df_split['original_split'], df_split['final_split'])
crosstab.to_csv(OUTPUT_DIR / '02_original_vs_final_split_crosstab.csv')
print('Saved: 02_original_vs_final_split_crosstab.csv')

# 10. Exclusion report
exclusion = pd.DataFrame([{
    'reason': 'unreadable',        'images_excluded': excluded_unreadable},
    {'reason': 'label_conflict',   'images_excluded': excluded_conflict},
    {'reason': 'other',            'images_excluded': 0},
    {'reason': 'final_included',   'images_excluded': len(df_split)},
    {'reason': 'total_excluded',   'images_excluded': len(df_raw) - len(df_split)},
])
exclusion.to_csv(OUTPUT_DIR / '02_exclusion_report.csv', index=False)
print('Saved: 02_exclusion_report.csv')


Saved: 02_group_safe_split_manifest.csv (2750 rows)
Saved: 02_final_split_summary.csv
Saved: 02_final_split_group_summary.csv
Saved: 02_original_vs_final_split_crosstab.csv
Saved: 02_exclusion_report.csv


## Contact Sheets


In [9]:
def make_contact_sheet(image_rows, title, save_path, cols=5, thumb=128):
    rows_needed = math.ceil(len(image_rows) / cols)
    fig, axes = plt.subplots(rows_needed, cols,
                             figsize=(cols * 2.2, rows_needed * 2.4))
    axes = np.array(axes).flatten()
    for i, (_, row) in enumerate(image_rows.iterrows()):
        ax = axes[i]
        try:
            img = Image.open(row['file_path']).convert('RGB')
            img.thumbnail((thumb, thumb))
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, 'ERR', ha='center', va='center',
                    transform=ax.transAxes, fontsize=8)
        label = f"{row.get('final_label','?')}\n{row.get('original_split','?')}\n{str(row.get('effective_group_id','?'))[:12]}"
        ax.set_title(label, fontsize=5, pad=2)
        ax.axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    fig.suptitle(title, fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.close()
    print(f'Contact sheet saved: {save_path}')


# Largest groups — top 20
top_groups = grp.nlargest(20, 'group_size')['effective_group_id'].tolist()
top_imgs = df[df['effective_group_id'].isin(top_groups)].groupby(
    'effective_group_id').head(5).reset_index(drop=True)
if len(top_imgs) > 0:
    make_contact_sheet(top_imgs, 'Largest groups (top 20, up to 5 images each)',
                       CONTACT_DIR / 'largest_groups.jpg')

# Label-conflict groups
if len(conflict_grp_ids) > 0:
    conf_imgs = df[df['effective_group_id'].isin(list(conflict_grp_ids)[:20])].groupby(
        'effective_group_id').head(5).reset_index(drop=True)
    make_contact_sheet(conf_imgs, 'Label-conflict groups',
                       CONTACT_DIR / 'label_conflict_groups.jpg')

# Groups spanning original splits
if spans > 0:
    span_grp_ids = grp[grp['spans_splits']]['effective_group_id'].tolist()[:20]
    span_imgs = df[df['effective_group_id'].isin(span_grp_ids)].groupby(
        'effective_group_id').head(5).reset_index(drop=True)
    make_contact_sheet(span_imgs, 'Groups spanning original train/valid/test',
                       CONTACT_DIR / 'split_spanning_groups.jpg')


Contact sheet saved: D:\DIABETES\diabetes_pipeline_outputs\contact_sheets_notebook02\largest_groups.jpg
Contact sheet saved: D:\DIABETES\diabetes_pipeline_outputs\contact_sheets_notebook02\split_spanning_groups.jpg


## Handoff Summary


In [10]:
split_counts  = df_split.groupby('final_split').size().to_dict()
class_counts  = df_split.groupby(['final_split','final_label']).size()
group_counts  = grp_clean.groupby('final_split').size().to_dict()

def gc(split, label):
    try: return int(class_counts.loc[split, label])
    except: return 0

handoff = [
    '================================================',
    'NOTEBOOK 02 — HANDOFF SUMMARY',
    '================================================',
    '',
    '1. Notebook status: completed',
    f'2. Input used: {MANIFEST_PATH}',
    '',
    '3. Outputs created:',
    *[f'   {OUTPUT_DIR / f}' for f in [
        '02_group_review_summary.csv','02_largest_groups.csv',
        '02_label_conflict_groups.csv','02_original_split_leakage_risk.csv',
        '02_group_safe_split_manifest.csv','02_final_split_summary.csv',
        '02_final_split_group_summary.csv','02_final_split_leakage_check.csv',
        '02_original_vs_final_split_crosstab.csv','02_exclusion_report.csv',
        '02_handoff_summary.txt'
    ]],
    f'   {CONTACT_DIR} (contact sheets)',
    '',
    '4. Group review:',
    f'   total images loaded                : {len(df)}',
    f'   total effective groups             : {len(grp)}',
    f'   single-image groups                : {single_img}',
    f'   multi-image groups                 : {multi_img}',
    f'   largest group size                 : {grp["group_size"].max()}',
    f'   label-conflict group count         : {conflict}',
    f'   groups spanning original splits    : {spans}',
    '',
    '5. Exclusions:',
    f'   unreadable excluded                : {excluded_unreadable}',
    f'   label-conflict images excluded     : {excluded_conflict}',
    f'   other excluded                     : 0',
    f'   final included images              : {len(df_split)}',
    f'   total excluded                     : {len(df_raw) - len(df_split)}',
    '',
    '6. Final split image counts:',
    f'   train : {split_counts.get("train", 0)}',
    f'   val   : {split_counts.get("val", 0)}',
    f'   test  : {split_counts.get("test", 0)}',
    '',
    '7. Final split class counts:',
    f'   train diabetes      : {gc("train","diabetes")}',
    f'   train non_diabetes  : {gc("train","non_diabetes")}',
    f'   val   diabetes      : {gc("val","diabetes")}',
    f'   val   non_diabetes  : {gc("val","non_diabetes")}',
    f'   test  diabetes      : {gc("test","diabetes")}',
    f'   test  non_diabetes  : {gc("test","non_diabetes")}',
    '',
    '8. Final split group counts:',
    f'   train groups : {group_counts.get("train", 0)}',
    f'   val   groups : {group_counts.get("val", 0)}',
    f'   test  groups : {group_counts.get("test", 0)}',
    '',
    '9. Leakage check:',
    f'   groups in multiple final splits    : {len(leaking)}',
    f'   status                             : {"PASS" if leakage_pass else "FAIL"}',
    '',
    '10. Important warnings:',
    f'    label-conflict groups             : {conflict}',
    f'    groups spanning original splits   : {spans}',
    f'    largest group size                : {grp["group_size"].max()} — review contact sheet',
    '',
    '11. Recommendation:',
    '    Review contact sheets in contact_sheets_notebook02/.',
    '    If leakage check = PASS and no label-conflict groups,',
    '    approve split and proceed to Notebook 03 (preprocessing).',
    '',
    'LIMITATION: Patient identifiers are unavailable. The final split is',
    'group-safe based on inferred effective_group_id. This reduces leakage',
    'risk but cannot guarantee perfect patient-level independence.',
    '================================================',
]

handoff_text = '\n'.join(handoff)
with open(OUTPUT_DIR / '02_handoff_summary.txt', 'w') as f:
    f.write(handoff_text)

print(handoff_text)


NOTEBOOK 02 — HANDOFF SUMMARY

1. Notebook status: completed
2. Input used: D:\DIABETES\diabetes_pipeline_outputs\01_master_manifest.csv

3. Outputs created:
   D:\DIABETES\diabetes_pipeline_outputs\02_group_review_summary.csv
   D:\DIABETES\diabetes_pipeline_outputs\02_largest_groups.csv
   D:\DIABETES\diabetes_pipeline_outputs\02_label_conflict_groups.csv
   D:\DIABETES\diabetes_pipeline_outputs\02_original_split_leakage_risk.csv
   D:\DIABETES\diabetes_pipeline_outputs\02_group_safe_split_manifest.csv
   D:\DIABETES\diabetes_pipeline_outputs\02_final_split_summary.csv
   D:\DIABETES\diabetes_pipeline_outputs\02_final_split_group_summary.csv
   D:\DIABETES\diabetes_pipeline_outputs\02_final_split_leakage_check.csv
   D:\DIABETES\diabetes_pipeline_outputs\02_original_vs_final_split_crosstab.csv
   D:\DIABETES\diabetes_pipeline_outputs\02_exclusion_report.csv
   D:\DIABETES\diabetes_pipeline_outputs\02_handoff_summary.txt
   D:\DIABETES\diabetes_pipeline_outputs\contact_sheets_notebook

## Final Validation Print


In [11]:
print('=' * 50)
print('NOTEBOOK 02 — FINAL VALIDATION')
print('=' * 50)
print(f'Total included images : {len(df_split)}')
print(f'Total excluded images : {len(df_raw) - len(df_split)}')
print()
print('Final split image counts:')
print(df_split['final_split'].value_counts().to_string())
print()
print('Final split class counts:')
print(df_split.groupby(['final_split','final_label']).size().to_string())
print()
print('Final split group counts:')
print(grp_clean['final_split'].value_counts().to_string())
print()
print(f'Effective groups in multiple final splits : {len(leaking)}')
print(f'Label-conflict groups found              : {conflict}')
print(f'Groups spanning original splits          : {spans}')
print(f'Final split leakage check                : {"PASS" if leakage_pass else "FAIL"}')
print()
print('All outputs saved to:', OUTPUT_DIR)
print('Contact sheets saved to:', CONTACT_DIR)
print('=' * 50)


NOTEBOOK 02 — FINAL VALIDATION
Total included images : 2750
Total excluded images : 0

Final split image counts:
final_split
train    1930
test      412
val       408

Final split class counts:
final_split  final_label 
test         diabetes        215
             non_diabetes    197
train        diabetes        961
             non_diabetes    969
val          diabetes        199
             non_diabetes    209

Final split group counts:
final_split
train    920
test     197
val      197

Effective groups in multiple final splits : 0
Label-conflict groups found              : 0
Groups spanning original splits          : 26
Final split leakage check                : PASS

All outputs saved to: D:\DIABETES\diabetes_pipeline_outputs
Contact sheets saved to: D:\DIABETES\diabetes_pipeline_outputs\contact_sheets_notebook02
